Additional End of week Exercise - week 2
Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

Technical Q&A Assistant - Week 2 Prototype
This prototype brings together Week 2 concepts:

Gradio UI for chat
Streaming responses
System prompt for technical expertise (coding assistant)
Model switching via LiteLLM / OpenRouter
Tool for documentation lookup

In [ ]:
# Imports
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from litellm import completion
import gradio as gr

In [ ]:
# Setup
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")
if openrouter_api_key:
    print(f"API key loaded (begins with {openrouter_api_key[:10]}...)")
else:
    print("WARNING: Set OPENROUTER_API_KEY or OPENAI_API_KEY in .env")

# OpenRouter client for tool calling (OpenRouter uses OpenAI-compatible API)
openrouter_url = "https://openrouter.ai/api/v1"
client = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

# Model choices: use OpenRouter model IDs
MODELS = {
    "GPT-4.1 Mini": "openai/gpt-4.1-mini",
    "Claude Sonnet 4.5": "anthropic/claude-sonnet-4.5",
    "Gemini 2.5 Flash": "google/gemini-2.5-flash",
    "Llama 3.2 (Ollama)": "ollama/llama3.2",
}
DEFAULT_MODEL = "GPT-4.1 Mini"

In [ ]:
# System prompt for technical expertise (from Week 1 Q&A assistant)
SYSTEM_PROMPT = """You are a helpful technical assistant and coding expert.
Explain technical concepts clearly and concisely. Include code examples when relevant.
Use markdown for formatting (headers, code blocks, lists).
If you use the look_up_documentation tool, incorporate the results into your answer.
When you don't know something, say so. Be accurate and avoid hallucinations."""

In [ ]:
# Tool: Documentation lookup 
DOC_SNIPPETS = {
    "list comprehension": "List comprehensions: [x**2 for x in range(10)]. Syntax: [expr for item in iterable if condition]",
    "yield": "yield produces a generator. Use 'yield from' to delegate to another generator. Generators are lazy and memory-efficient.",
    "async": "async def defines a coroutine. Use 'await' inside it. Run with asyncio.run() or an event loop.",
    "decorator": "Decorators wrap functions. @decorator def fn(): pass. A decorator is a function that returns a function.",
    "context manager": "Use 'with' for context managers. Define __enter__ and __exit__, or use @contextmanager with yield.",
}

def look_up_documentation(topic: str) -> str:
    """Look up documentation for a programming topic. Returns a brief reference."""
    key = topic.lower().strip()
    return DOC_SNIPPETS.get(key, f"No cached documentation for '{topic}'. Check official docs: Python—docs.python.org, JavaScript—developer.mozilla.org")

# Tool definition for the LLM (OpenAI function-calling format)
doc_tool = {
    "type": "function",
    "function": {
        "name": "look_up_documentation",
        "description": "Look up programming documentation for a topic (e.g. list comprehension, yield, async, decorator, context manager).",
        "parameters": {
            "type": "object",
            "properties": {"topic": {"type": "string", "description": "The programming concept to look up"}},
            "required": ["topic"],
            "additionalProperties": False,
        },
    },
}
tools = [{"type": "function", "function": doc_tool["function"]}]

In [ ]:
def handle_tool_calls(message):
    """Execute tool calls and return tool messages (from day5)."""
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "look_up_documentation":
            args = json.loads(tool_call.function.arguments)
            result = look_up_documentation(args.get("topic", ""))
            responses.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
    return responses

def _assistant_message_to_dict(msg):
    """Convert assistant message with tool_calls to API format."""
    return {
        "role": "assistant",
        "content": msg.content or "",
        "tool_calls": [
            {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in msg.tool_calls
        ],
    }

In [ ]:
def chat(message, history, model_name, enable_tools=True):
    """Chat callback: streaming, system prompt, model switching, tools."""
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]
    model_id = MODELS.get(model_name, MODELS[DEFAULT_MODEL])
    use_tools = enable_tools and not model_id.startswith("ollama/")

    if use_tools:
        
        response = client.chat.completions.create(model=model_id, messages=messages, tools=tools)
        while response.choices[0].finish_reason == "tool_calls":
            msg = response.choices[0].message
            messages.append(_assistant_message_to_dict(msg))
            messages.extend(handle_tool_calls(msg))
            response = client.chat.completions.create(model=model_id, messages=messages, tools=tools)
        reply = response.choices[0].message.content or ""
        yield reply
    else:
        # Streaming: use LiteLLM for all models
        stream = completion(model=model_id, messages=messages, stream=True, api_key=openrouter_api_key)
        reply = ""
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                reply += chunk.choices[0].delta.content
                yield reply

In [ ]:
# Gradio UI
def add_user_msg(msg, hist):
    """Prepend user message to history for display."""
    hist = list(hist) if hist else []
    if msg:
        hist.append({"role": "user", "content": msg})
    return "", hist

def _to_msg(h):
    """Normalize history item to {role, content} dict."""
    if isinstance(h, dict):
        return {"role": h.get("role", "user"), "content": h.get("content", "") or ""}
    return {"role": "user", "content": str(h)}

def stream_chat(hist, model, use_tools):
    """Stream chat response into chatbot. Yields full history + growing assistant reply."""
    hist = list(hist or [])
    if not hist:
        return
    last = _to_msg(hist[-1])
    if last["role"] != "user":
        return
    msg = last["content"]
    hist_prev = [_to_msg(h) for h in hist[:-1]]
    base = [_to_msg(h) for h in hist]
    for reply in chat(msg, hist_prev, model, use_tools):
        yield base + [{"role": "assistant", "content": reply}]

with gr.Blocks(title="Technical Q&A Assistant") as demo:
    gr.Markdown("Ask coding questions. Try: 'Explain list comprehension' or 'What does yield do?'")
    chatbot = gr.Chatbot(height=500, type="messages")
    with gr.Row():
        tb = gr.Textbox(label="Your question", placeholder="Ask a technical question...", scale=4)
        submit_btn = gr.Button("Send", variant="primary", scale=1)
    with gr.Row():
        model_dd = gr.Dropdown(choices=list(MODELS.keys()), value=DEFAULT_MODEL, label="Model")
        tools_cb = gr.Checkbox(value=True, label="Use documentation tool")
    clear_btn = gr.Button("Clear")

    def clear():
        return [], ""

    tb.submit(add_user_msg, [tb, chatbot], [tb, chatbot]).then(
        stream_chat, [chatbot, model_dd, tools_cb], chatbot
    )
    submit_btn.click(add_user_msg, [tb, chatbot], [tb, chatbot]).then(
        stream_chat, [chatbot, model_dd, tools_cb], chatbot
    )
    clear_btn.click(clear, None, [chatbot, tb])

In [ ]:
demo.launch()